# NB04B — ROI-restricted seasonal S2 LOSO (V3_1)

**Scientific design:** reuse NB04 full-mosaic cached UAV aggregates; resample **inside ROI.shp** at 75/100/125 m. Evaluate 5 mosaic groups (MOS1–2 and MOS5–6 separately) and a stricter 4-forest scheme holding both MOS mosaics out together. Predictions refer to UAV-model-derived targets. Missing quarterly products stop execution with explicit path errors. No new Earth Engine extraction.

**Important:** requires correctly exported quarterly rasters from NB04A gridfix, not the older mis-shaped files.

In [ ]:
# 00 — USER SETTINGS (edit only here)
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
FINAL=ROOT/"Final"
ROI_PATH=ROOT/"data/Orthoo_RGB/ROI.shp"
SCALES=[75,100,125]
TARGETS=["mean_probability"]  # optional: add "cover_fraction_p05"
FEATURE_SETS={
 "EMB":["EMB_ANNUAL"],
 "QUARTERLY":["S2_QUARTERLY_3YR"],
 "EMB_QUARTERLY":["EMB_ANNUAL","S2_QUARTERLY_3YR"],
 "EMB_ANNUAL_QUARTERLY":["EMB_ANNUAL","RAW_ANNUAL","S2_QUARTERLY_3YR"]
}
CV_SCHEMES=["FIVE_MOSAICS","FOUR_FORESTS"]
MAX_CELLS_PER_MOSAIC=8000
MIN_ROI_COVERAGE=0.90    # fractional pixel coverage inside ROI
ROI_SUBPIXELS=4          # 4x4 supersampling
MIN_UAV_COVERAGE=0.90
MIN_QUARTER_CLEAR=1
SEED=20261022
WEIGHT_STRENGTHS=[0,2]
HIGH_COVER_THRESHOLD=0.40
FORCE_SAMPLES=False
FORCE_FOLDS=False
SAVE_PLOTS=True


In [ ]:
# 01 — Dependencies, input manifest, reuse cache
%pip -q install rasterio geopandas pyarrow scikit-learn matplotlib
import json,math,gc,time,hashlib
import numpy as np,pandas as pd,geopandas as gpd,rasterio
from rasterio.features import rasterize
from rasterio.warp import reproject,Resampling
from affine import Affine
from shapely.geometry import box
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import r2_score
from IPython.display import display
import matplotlib.pyplot as plt
from google.colab import drive
if not Path("/content/drive/MyDrive").exists(): drive.mount("/content/drive")
V3=FINAL/"Outputs/V3/04_satellite_Musanga/features"
OLD=FINAL/"Outputs/V3_1/04_satellite_full_warmstart"
AGG=OLD/"cache/aggregates"
QUART=FINAL/"Outputs/V3_1/04A_quarterly_3year_gridfix"
OUT=FINAL/"Outputs/V3_1/04B_roi_quarterly_LOSO"
SAMPLES=OUT/"samples";FOLDS=OUT/"folds";FIG=OUT/"figures"
for p in [SAMPLES,FOLDS,FIG]:p.mkdir(parents=True,exist_ok=True)
YEARS={"Baego":2023,"MOS":2024,"Yangambi":2024,"Yoko":2020}
WINDOWS={"Baego":(2021,2023),"MOS":(2022,2024),"Yangambi":(2022,2024),"Yoko":(2018,2020)}
MOSAICS={"Baego":["Baego.tif"],"MOS":["Mos_1_2_transparent_mosaic_group1_georef.tif",
"Mos_5_6_transparent_mosaic_group1_georef.tif"],"Yangambi":["Yangambi_georef.tif"],"Yoko":["Yoko_kvo_georef.tif"]}
GROUPS={"Baego":"Baego","Mos_1_2_transparent_mosaic_group1_georef":"MOS1_2",
"Mos_5_6_transparent_mosaic_group1_georef":"MOS5_6","Yangambi_georef":"Yangambi","Yoko_kvo_georef":"Yoko"}
manifest=FINAL/"Outputs/V3/00_input_audit/00_input_manifest.json"
if not ROI_PATH.exists() and manifest.exists():ROI_PATH=Path(json.loads(manifest.read_text())["roi"])
assert ROI_PATH.exists(),ROI_PATH
ROI=gpd.read_file(ROI_PATH)
assert ROI.crs is not None and len(ROI)>0,"ROI CRS or geometries missing"
print("ROI",ROI_PATH,"polygons",len(ROI),flush=True)
def path_feature(site,family):
    if family=="S2_QUARTERLY_3YR":
        a,b=WINDOWS[site];return QUART/f"{site}_S2_QUARTERLY_3YR_{a}_{b}_10m.tif"
    return V3/family/f"{site}_{family}_{YEARS[site]}_10m.tif"
def agg_path(mosaic,scale):return AGG/f"{Path(mosaic).stem}_{scale}m.tif"
for site,names in MOSAICS.items():
    for family in sorted(set(sum(FEATURE_SETS.values(),[]))):
        p=path_feature(site,family)
        assert p.exists(),f"Missing feature: {p}. Finish NB04A and QA before NB04B."
    for name in names:
        for scale in SCALES:
            assert agg_path(name,scale).exists(),f"Missing cached UAV aggregate: {agg_path(name,scale)}"
print("ALL source products found. Existing UAV aggregates will NOT be rebuilt.",flush=True)


In [ ]:
# 02 — Geometry grids, fractional ROI cover and cached sampling
def grid_for(site,scale):
    with rasterio.open(path_feature(site,"RAW_ANNUAL")) as ds:
        assert ds.crs is not None and abs(ds.transform.a-10)<.001 and abs(ds.transform.e+10)<.001
        f=scale/10
        return dict(height=math.ceil(ds.height/f),width=math.ceil(ds.width/f),
                    transform=ds.transform*Affine.scale(f,f),crs=ds.crs)
def warp_band(ds,index,g,mode=Resampling.average):
    a=np.full((g["height"],g["width"]),-9999.,dtype=np.float32)
    reproject(source=rasterio.band(ds,index),destination=a,src_nodata=ds.nodata,
              dst_transform=g["transform"],dst_crs=g["crs"],dst_nodata=-9999.,
              resampling=mode,num_threads=2)
    a[a==-9999]=np.nan
    return a
def roi_fraction(g):
    k=ROI_SUBPIXELS
    local=ROI.to_crs(g["crs"])
    fine=rasterize([(geom,1) for geom in local.geometry if geom is not None and not geom.is_empty],
           out_shape=(g["height"]*k,g["width"]*k),
           transform=g["transform"]*Affine.scale(1/k,1/k),fill=0,all_touched=False,dtype="uint8")
    return fine.reshape(g["height"],k,g["width"],k).mean(axis=(1,3))
def cube_on_grid(site,g):
    cubes=[];cols=[];quarter_clear=[]
    for family in ["RAW_ANNUAL","EMB_ANNUAL","S2_QUARTERLY_3YR"]:
        with rasterio.open(path_feature(site,family)) as ds:
            count_expected=44 if family=="S2_QUARTERLY_3YR" else None
            if count_expected:assert ds.count==count_expected,(site,ds.count)
            assert ds.crs==g["crs"],f"CRS mismatch {site} {family}"
            for b in range(1,ds.count+1):
                mode=Resampling.nearest if family=="S2_QUARTERLY_3YR" and b%11==0 else Resampling.average
                band=warp_band(ds,b,g,mode=mode)
                if family=="S2_QUARTERLY_3YR" and b%11==0:quarter_clear.append(band)
                cubes.append(band);cols.append(f"{family}__{b:02d}")
    return np.stack(cubes),cols,np.stack(quarter_clear)
SAMPLE_FRAMES={}
for scale in SCALES:
    cache=SAMPLES/f"roi_cells_{scale}m.parquet"
    if cache.exists() and not FORCE_SAMPLES:
        df=pd.read_parquet(cache)
        SAMPLE_FRAMES[scale]=df;print("ROI SAMPLE CACHE",scale,len(df),flush=True);continue
    rows=[]
    for site,names in MOSAICS.items():
        g=grid_for(site,scale)
        fraction=roi_fraction(g)
        cube,cols,clear=cube_on_grid(site,g)
        # All four quarterly clear count bands must satisfy minimum count.
        feature_valid=np.isfinite(cube).all(axis=0)&np.isfinite(clear).all(axis=0)&(clear>=MIN_QUARTER_CLEAR).all(axis=0)
        for name in names:
            with rasterio.open(agg_path(name,scale)) as ds:
                assert ds.crs==g["crs"] and ds.transform.almost_equals(g["transform"],precision=1e-6)
                y1,y2,uav_valid=ds.read()
            ok=(fraction>=MIN_ROI_COVERAGE)&(uav_valid>=MIN_UAV_COVERAGE)&feature_valid
            ok &= np.isfinite(y1)&np.isfinite(y2)&(y1>=0)&(y1<=1)&(y2>=0)&(y2<=1)
            rr,cc=np.where(ok)
            rng=np.random.default_rng(SEED+sum(map(ord,name))+scale)
            take=rng.choice(len(rr),min(len(rr),MAX_CELLS_PER_MOSAIC),replace=False)
            rr,cc=rr[take],cc[take]
            df=pd.DataFrame(cube[:,rr,cc].T,columns=cols)
            df.insert(0,"site",site);df.insert(1,"group",GROUPS[Path(name).stem])
            df.insert(2,"mosaic",name);df.insert(3,"row",rr);df.insert(4,"col",cc)
            df["roi_fraction"]=fraction[rr,cc]
            df["mean_probability"]=y1[rr,cc];df["cover_fraction_p05"]=y2[rr,cc]
            rows.append(df)
            print("ROI SAMPLES",scale,site,GROUPS[Path(name).stem],len(df),"candidate",len(np.where(ok)[0]),flush=True)
        del cube,clear;gc.collect()
    frame=pd.concat(rows,ignore_index=True)
    assert frame.group.nunique()==5,f"Only {frame.group.nunique()} groups with sampled cells"
    frame.to_parquet(cache,index=False);SAMPLE_FRAMES[scale]=frame
    print("SAVED",cache,len(frame),flush=True)


In [ ]:
# 03 — Five-mosaic versus four-forest LOSO; checkpoint each fold
def scores(y,p):
    y=np.asarray(y,float);p=np.asarray(p,float)
    high=y>=HIGH_COVER_THRESHOLD
    return dict(n=len(y),R2=float(r2_score(y,p)) if len(y)>1 else np.nan,
                MAE_pp=float(100*np.mean(abs(p-y))),RMSE_pp=float(100*np.sqrt(np.mean((p-y)**2))),
                bias_pp=float(100*np.mean(p-y)),n_high=int(high.sum()),
                high_bias_pp=float(100*np.mean((p-y)[high])) if high.any() else np.nan,
                high_recall=float(np.mean(p[high]>=HIGH_COVER_THRESHOLD)) if high.any() else np.nan)
records=[]
for scale,df in SAMPLE_FRAMES.items():
    for scheme in CV_SCHEMES:
        fold_col="group" if scheme=="FIVE_MOSAICS" else "site"
        for holdout in sorted(df[fold_col].unique()):
            train=df[df[fold_col]!=holdout];test=df[df[fold_col]==holdout]
            if len(train)<100 or len(test)<10:
                print("SKIP insufficient samples",scale,scheme,holdout,len(train),len(test),flush=True)
                continue
            for name,families in FEATURE_SETS.items():
                cols=[col for col in df.columns if any(col.startswith(f+"__") for f in families)]
                xtr=train[cols].to_numpy(np.float32);xte=test[cols].to_numpy(np.float32)
                assert np.isfinite(xtr).all() and np.isfinite(xte).all()
                for target in TARGETS:
                    ytr=train[target].to_numpy(np.float32);yte=test[target].to_numpy(np.float32)
                    for strength in WEIGHT_STRENGTHS:
                        key=f"{scheme}_{scale}m_{holdout}_{name}_{target}_w{strength}"
                        dest=FOLDS/f"{key}.parquet"
                        if dest.exists() and not FORCE_FOLDS:
                            fold=pd.read_parquet(dest);print("CACHE",key,flush=True)
                        else:
                            start=time.perf_counter()
                            weights=1+strength*np.clip((ytr-.20)/.40,0,1)
                            model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
                                max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
                                early_stopping=False,random_state=SEED)
                            model.fit(xtr,ytr,sample_weight=weights)
                            pred=np.clip(model.predict(xte),0,1)
                            fold=test[["site","group","mosaic","row","col"]].copy()
                            fold["observed"]=yte;fold["predicted"]=pred
                            fold.to_parquet(dest,index=False)
                            print("DONE",key,"seconds",round(time.perf_counter()-start,1),flush=True)
                        assert len(fold)==len(test),(key,"stale cache")
                        rec=dict(scheme=scheme,scale_m=scale,holdout=holdout,
                            predictors=name,target=target,weight=strength,**scores(fold.observed,fold.predicted))
                        records.append(rec)
        pd.DataFrame(records).to_csv(OUT/"fold_metrics_progress.csv",index=False)
results=pd.DataFrame(records);results.to_csv(OUT/"fold_metrics.csv",index=False)
display(results.round(3))


In [ ]:
# 04 — Pooled and equal-group results, MOS tail, separate CV domains
summary=[]
for (scheme,scale,predictors,target,weight),g in results.groupby(
    ["scheme","scale_m","predictors","target","weight"]):
    frames=[]
    for heldout in g.holdout:
        p=FOLDS/f"{scheme}_{scale}m_{heldout}_{predictors}_{target}_w{weight}.parquet"
        frames.append(pd.read_parquet(p))
    d=pd.concat(frames,ignore_index=True)
    y=d.observed.to_numpy(float);pred=d.predicted.to_numpy(float)
    n=d.groupby("group" if scheme=="FIVE_MOSAICS" else "site").size()
    grouping="group" if scheme=="FIVE_MOSAICS" else "site"
    w=d[grouping].map(lambda s:1/n.loc[s]).to_numpy(float);w/=w.sum()
    ybar=np.dot(w,y);sst=np.dot(w,(y-ybar)**2);mse=np.dot(w,(y-pred)**2)
    grouped_means=d.groupby(grouping).observed.mean()
    site_center=y-d[grouping].map(grouped_means).to_numpy(float)
    within=np.dot(w,site_center**2)
    mos=d[d.site=="MOS"];hi=mos.observed>=HIGH_COVER_THRESHOLD
    summary.append(dict(scheme=scheme,scale_m=scale,predictors=predictors,target=target,
        weight=weight,n=len(d),pooled_R2=r2_score(y,pred),
        balanced_R2=1-mse/sst if sst>0 else np.nan,
        within_group_R2=1-mse/within if within>0 else np.nan,
        macro_MAE_pp=100*np.dot(w,abs(y-pred)),
        MOS_high_n=int(hi.sum()),
        MOS_high_bias_pp=100*(mos.predicted[hi]-mos.observed[hi]).mean() if hi.any() else np.nan,
        MOS_high_recall=float((mos.predicted[hi]>=HIGH_COVER_THRESHOLD).mean()) if hi.any() else np.nan))
summary=pd.DataFrame(summary)
summary.to_csv(OUT/"global_metrics.csv",index=False)
for scheme in CV_SCHEMES:
    print("\n",scheme,flush=True)
    display(summary[summary.scheme==scheme].sort_values("macro_MAE_pp").round(3))
print("Output",OUT,flush=True)


In [ ]:
# 05 — MOS high-cover scatterplots, held-out predictions only
if SAVE_PLOTS:
    for scheme in CV_SCHEMES:
        subset=summary[(summary.scheme==scheme)&(summary.target=="mean_probability")].sort_values("macro_MAE_pp")
        if not len(subset):continue
        winner=subset.iloc[0]
        scale=int(winner.scale_m);name=winner.predictors;wt=int(winner.weight)
        fig,axes=plt.subplots(1,2,figsize=(11,5),constrained_layout=True)
        for ax,group in zip(axes,["MOS1_2","MOS5_6"]):
            heldout=group if scheme=="FIVE_MOSAICS" else "MOS"
            f=FOLDS/f"{scheme}_{scale}m_{heldout}_{name}_mean_probability_w{wt}.parquet"
            if not f.exists():continue
            d=pd.read_parquet(f);d=d[d.group==group]
            ax.scatter(d.observed*100,d.predicted*100,s=7,alpha=.25,rasterized=True)
            ax.plot([0,100],[0,100],"--",color="grey")
            ax.axvline(HIGH_COVER_THRESHOLD*100,linestyle=":",color="grey")
            ax.set(xlim=(0,100),ylim=(0,100),title=f"{scheme}: {group} (n={len(d)})",
                   xlabel="UAV-derived mean probability (%)",ylabel="Satellite predicted (%)")
        fig.suptitle(f"{name} | {scale} m | weighting {wt}")
        fig.savefig(FIG/f"MOS_{scheme}_best.png",dpi=175)
        plt.show();plt.close(fig)


## Scientific caveats

- Five-mosaic LOSO can train on the *other* MOS mosaic. Four-forest LOSO holds both MOS mosaics out together and is the defensible strict geographic transfer test.
- Cached UAV aggregates were computed on full image grids; this notebook restricts their *sampled cells* to ROI with supersampled fractional coverage.
- Quarterly composites may include satellite observations after UAV flights. Temporal causality is not established.
- High-cover MOS diagnostics use the UAV model's pseudo-labels, not field-measured abundance.
- Comparisons use common complete-case features, which can reduce sample size; inspect per-mosaic sample counts.